In [ ]:
#*****************************************Import********************************************
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder 
import warnings
import time
from sklearn.model_selection import cross_val_score
from sklearn import metrics
import re
from sklearn import preprocessing
from sklearn.linear_model import LinearRegression 
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier      
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier      
import seaborn as sns
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
df = pd.read_csv("StealthPhisher.csv")
LABEL = df.iloc[:,-1:].columns[0]

fs = ['phishid', 'url_len', 'url_complexity', 'url_domainlen', 'isdomainanip',
'tldlen', 'url_letter', 'url_letter_ration', 'url_digit',
 'url_digit_ration', 'url_equal', 'url_qmark', 'url_amp',
 'url_otherspecialchar', 'url_spclchar_ration', 'url_ssl', 'unreachable',
 'lineofcode', 'hastitle', 'favicon', 'robots', 'responsive',
 'urlredirect', 'selfredirect', 'description', 'formsubmitexternal',
 'socialnet', 'submitbutton', 'hiddenfield', 'passwordfield', 'bank',
 'pay', 'hascopyrightinfo', 'number_of_img', 'number_of_css',
 'number_of_js', 'number_of_selfhref', 'number_of_extref',
 'number_of_iframe',LABEL]

df = pd.DataFrame(df[fs]).copy()

print('Dataset Shape: ', df.shape)
df.iloc[:,-1:].value_counts()

In [ ]:
print("Starting Data Transformation...\n")
df = df[df[LABEL] == 0]
myTrain, myTest=train_test_split(df,test_size=0.3, random_state=15)

scaler = StandardScaler()

cols = myTrain.select_dtypes(include=['float64','int64']).columns
scalerTrain = scaler.fit_transform(myTrain.select_dtypes(include=['float64','int64']))
scalerTest = scaler.fit_transform(myTest.select_dtypes(include=['float64','int64']))

scalerTrainDF = pd.DataFrame(scalerTrain, columns = cols)
scalerTestDF = pd.DataFrame(scalerTest, columns = cols)

oneHotEncoder = OneHotEncoder() 
trainLabel = myTrain.iloc[:,-1:].values.reshape(-1,1)
trainLabel = oneHotEncoder.fit_transform(trainLabel).toarray()
testLabel = myTest.iloc[:,-1:].values.reshape(-1,1)
testLabel = oneHotEncoder.fit_transform(testLabel).toarray()

X_TrainMaster=scalerTrainDF
Y_TrainMaster=trainLabel[:,0]
X_TestMaster=scalerTestDF
Y_TestMaster=testLabel[:,0]

if(df[LABEL].dtypes != 'O'):
    X_TrainMaster = X_TrainMaster.iloc[:,:-1]
    X_TestMaster = X_TestMaster.iloc[:,:-1]
    
X_Train = None
Y_Train = None
X_Test = None
Y_Test = None

X_Train = pd.DataFrame(X_TrainMaster).copy()
Y_Train = pd.DataFrame(Y_TrainMaster).copy()
X_Test = pd.DataFrame(X_TestMaster).copy() 
Y_Test = pd.DataFrame(Y_TestMaster).copy()
DataCleanEND = time.time()
print("Finished Data Transformation.")

In [ ]:
LightGBM = LGBMClassifier(learning_rate=3)
LightGBM.fit(X_Train, Y_Train)
scoreLGBM = np.round(LightGBM.feature_importances_,3)
iLGBM = pd.DataFrame({'Features':X_Train.columns,'importance':scoreLGBM})
plt.rcParams['figure.figsize'] = (12, 4)
iLGBM = iLGBM.set_index('Features')
iLGBM.plot.bar();

In [ ]:
RF = RandomForestClassifier ();
RF.fit(X_Train, Y_Train);
scoreRF = np.round(RF.feature_importances_,3)
iRF = pd.DataFrame({'Features':X_Train.columns,'importance':scoreRF})
plt.rcParams['figure.figsize'] = (12, 4)
iRF = iRF.set_index('Features')
iRF.plot.bar();